# Stock Return Predictor
## Notebook 2: Feature Engineering

**Project:** Predicting whether a stock will outperform the S&P 500 over the next month  
**Author:** Romir Kapoor 
**Date:** April 2026  

---

### Objective
Transform raw price data into meaningful features that capture momentum, trend, 
volatility, and relative performance. Then construct the target variable — whether 
a stock beat the S&P 500 the following month.

### Contents
1. Imports & Setup
2. Load Price Data
3. Calculate Monthly Returns
4. Build Features
5. Build Target Variable
6. Combine & Clean
7. Save to Disk

## 1. Imports & Setup

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

## 2. Load Price Data

In [23]:
DATA_DIR = "../data"

prices_df = pd.read_csv(f"{DATA_DIR}/prices.csv", index_col=0, parse_dates=True)
print(f"Loaded prices — shape: {prices_df.shape}")
prices_df.head()

Loaded prices — shape: (1760, 20)


,AAPL,ABT,AMZN,BAC,C,COP,CVX,EOG,GOOGL,GS,JNJ,JPM,META,MRK,MSFT,PFE,SLB,UNH,WFC,XOM
Date,,,,,,,,,,,,,,,,,,,,
2018-01-02,40.304176,50.506081,59.450500,24.637495,56.579605,42.354919,89.452377,80.424561,53.220631,213.144241,110.764206,85.901283,180.007385,41.593494,78.870361,23.830420,54.936550,193.090454,48.653320,58.580410
2018-01-03,40.297153,50.617756,60.209999,24.555096,56.754620,43.135872,90.104439,81.623909,54.128628,211.160156,111.822266,85.988808,183.232086,41.534306,79.237404,24.006990,56.366863,195.115997,49.027637,59.730938
2018-01-04,40.484337,50.531853,60.479500,24.876459,57.454605,43.633533,89.823997,81.638649,54.338882,214.111252,111.814301,87.220634,182.894745,42.207558,79.934837,24.059305,57.765579,195.962921,49.640896,59.813610
2018-01-05,40.945255,50.677902,61.457001,24.991812,57.378536,43.549320,89.676735,82.536354,55.059433,213.019180,112.737144,86.660713,185.395126,42.163174,80.925858,24.105080,58.160679,199.699692,49.975372,59.765408
2018-01-08,40.793182,50.531853,62.343498,24.818775,56.708942,43.993389,90.118446,82.933693,55.253830,209.926285,112.880341,86.788719,186.813980,41.919029,81.008438,23.836956,59.108944,196.233566,49.409920,60.034081


## 3. Calculate Monthly Returns
Resample daily prices to month-end, then compute month-over-month returns.

In [17]:
# Resample to month-end prices
monthly_prices = prices_df.resample("ME").last()
print(f"Monthly prices shape: {monthly_prices.shape}")
monthly_prices.head()

Monthly prices shape: (84, 20)


,AAPL,ABT,AMZN,BAC,C,COP,CVX,EOG,GOOGL,GS,JNJ,JPM,META,MRK,MSFT,PFE,SLB,UNH,WFC,XOM
Date,,,,,,,,,,,,,,,,,,,,
2018-01-31,39.174084,53.656605,72.544502,26.367886,59.714462,45.026993,87.888809,84.741737,58.626446,223.331711,109.936829,92.523941,185.434814,43.835201,87.184082,24.222792,58.144863,206.727982,52.388531,60.144329
2018-02-28,41.845169,52.076954,75.622498,26.450291,57.672401,41.805431,79.253578,74.734840,54.743546,219.811966,103.987442,92.387978,176.931549,40.113834,86.450424,23.965387,52.236538,197.455856,46.796371,52.713753
2018-03-31,39.416027,51.723038,72.366997,24.804379,51.568272,45.638802,80.754814,77.571854,51.431911,210.557327,102.602325,87.964546,158.545807,40.651192,84.145569,23.424170,51.552143,187.457855,41.989346,51.927299
2018-04-30,38.824017,50.419434,78.306503,24.746485,52.156536,50.418995,88.593773,87.224403,50.511524,199.246155,101.273247,87.453926,170.660736,43.934921,86.219925,24.163389,54.560280,207.079605,41.628819,54.112682
2018-05-31,44.070076,53.368439,81.481003,24.116787,51.190571,52.087902,88.787727,86.958672,54.549160,189.505646,96.483139,86.030914,190.286728,44.427479,91.519638,23.946074,54.647816,211.555725,43.580868,57.114780


In [18]:
# Calculate simple monthly returns
monthly_returns = monthly_prices.pct_change()
print(f"Monthly returns shape: {monthly_returns.shape}")
monthly_returns.head()

Monthly returns shape: (84, 20)


,AAPL,ABT,AMZN,BAC,C,COP,CVX,EOG,GOOGL,GS,JNJ,JPM,META,MRK,MSFT,PFE,SLB,UNH,WFC,XOM
Date,,,,,,,,,,,,,,,,,,,,
2018-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-02-28,0.068185,-0.029440,0.042429,0.003125,-0.034197,-0.071547,-0.098252,-0.118087,-0.066231,-0.015760,-0.054116,-0.001469,-0.045856,-0.084894,-0.008415,-0.010627,-0.101614,-0.044852,-0.106744,-0.123546
2018-03-31,-0.058051,-0.006796,-0.043049,-0.062227,-0.105841,0.091696,0.018942,0.037961,-0.060494,-0.042103,-0.013320,-0.047879,-0.103914,0.013396,-0.026661,-0.022583,-0.013102,-0.050634,-0.102722,-0.014919
2018-04-30,-0.015020,-0.025204,0.082075,-0.002334,0.011407,0.104740,0.097071,0.124434,-0.017895,-0.053720,-0.012954,-0.005805,0.076413,0.080778,0.024652,0.031558,0.058351,0.104673,-0.008586,0.042085
2018-05-31,0.135124,0.058489,0.040539,-0.025446,-0.018521,0.033101,0.002189,-0.003047,0.079935,-0.048887,-0.047299,-0.016272,0.115000,0.011211,0.061467,-0.008994,0.001604,0.021615,0.046892,0.055479


## 4. Build Features
Momentum, moving average ratios, and volatility for each stock.

In [24]:
def compute_features(prices_daily, monthly_prices):
    features_list = []

    for date in monthly_prices.index[6:]:  # start after 6 months for lookback
        row = {"date": date}

        for ticker in monthly_prices.columns:
            # get daily prices up to this date
            hist = prices_daily[ticker].loc[:date]

            if len(hist) < 126:
                continue

            price_today = hist.iloc[-1]

            # --- Momentum ---
            ret_1m  = (price_today / hist.iloc[-21]  - 1) if len(hist) >= 21  else np.nan
            ret_3m  = (price_today / hist.iloc[-63]  - 1) if len(hist) >= 63  else np.nan
            ret_6m  = (price_today / hist.iloc[-126] - 1) if len(hist) >= 126 else np.nan

            # --- Moving average ratios ---
            ma_20 = hist.iloc[-20:].mean()
            ma_50 = hist.iloc[-50:].mean()
            ma_ratio_20 = price_today / ma_20 - 1
            ma_ratio_50 = price_today / ma_50 - 1

            # --- Volatility ---
            daily_rets = hist.pct_change().iloc[-20:]
            volatility_20 = daily_rets.std() * np.sqrt(252)  # annualized

            row[f"{ticker}_ret_1m"]       = ret_1m
            row[f"{ticker}_ret_3m"]       = ret_3m
            row[f"{ticker}_ret_6m"]       = ret_6m
            row[f"{ticker}_ma_ratio_20"]  = ma_ratio_20
            row[f"{ticker}_ma_ratio_50"]  = ma_ratio_50
            row[f"{ticker}_volatility"]   = volatility_20

        features_list.append(row)

    features_df = pd.DataFrame(features_list).set_index("date")
    return features_df

features_df = compute_features(prices_df, monthly_prices)
print(f"Features shape: {features_df.shape}")
features_df.head()

Features shape: (78, 120)


,AAPL_ret_1m,AAPL_ret_3m,AAPL_ret_6m,AAPL_ma_ratio_20,AAPL_ma_ratio_50,AAPL_volatility,ABT_ret_1m,ABT_ret_3m,ABT_ret_6m,ABT_ma_ratio_20,...,WFC_ret_6m,WFC_ma_ratio_20,WFC_ma_ratio_50,WFC_volatility,XOM_ret_1m,XOM_ret_3m,XOM_ret_6m,XOM_ma_ratio_20,XOM_ma_ratio_50,XOM_volatility
date,,,,,,,,,,,,,,,,,,,,,
2018-07-31,0.016615,0.081859,0.145569,-0.000934,0.005789,0.152962,0.079256,0.138022,0.064213,0.034452,...,-0.117276,0.007120,0.030355,0.201682,-0.003058,0.072086,-0.047215,-0.011922,-0.003367,0.155567
2018-08-31,0.098266,0.181668,0.297952,0.061912,0.139644,0.121025,0.024682,0.065214,0.108141,0.024752,...,0.039977,-0.001907,0.025242,0.090618,0.009936,0.003928,0.074008,0.010195,-0.003456,0.140775
2018-09-30,0.003155,0.210234,0.350468,0.015239,0.059595,0.225772,0.094598,0.208028,0.254991,0.067590,...,0.024256,-0.056094,-0.081397,0.155354,0.056018,0.050598,0.156592,0.019192,0.045613,0.138768
2018-10-31,-0.056923,0.055953,0.199339,-0.003106,-0.010474,0.344657,-0.039097,0.061221,0.183769,-0.000563,...,0.023078,0.012016,-0.026778,0.276419,-0.075102,0.003764,0.057449,-0.021347,-0.032039,0.221113
2018-11-30,-0.193578,-0.215260,-0.069724,-0.057841,-0.141543,0.444800,0.055595,0.107949,0.184970,0.045225,...,0.011368,0.023437,0.030923,0.157908,-0.004513,0.000198,0.005630,0.006506,-0.015383,0.231889


## 5. Build Target Variable
For each stock and each month, did it outperform SPY the following month?  
1 = outperformed, 0 = underperformed.

In [27]:
import yfinance as yf

# Pull SPY as benchmark
spy_raw = yf.download("SPY", start="2018-01-01", end="2024-12-31", auto_adjust=True)
spy_monthly = spy_raw["Close"].resample("ME").last()
spy_returns = spy_monthly.pct_change()

spy_returns.head()

[*********************100%***********************]  1 of 1 completed


Ticker,SPY
Date,
2018-01-31,NaN
2018-02-28,-0.036361
2018-03-31,-0.027411
2018-04-30,0.005168
2018-05-31,0.024309


In [31]:
# Build target: did stock beat SPY next month?
target_list = []

for date in features_df.index:
    row = {"date": date}
    
    # find next month's return
    future_dates = spy_returns.index[spy_returns.index > date]
    if len(future_dates) == 0:
        continue
    next_month = future_dates[0]
    
    # extract scalar value from Series
    spy_next = spy_returns.loc[next_month]
    if hasattr(spy_next, 'item'):
        spy_next = spy_next.item()

    for ticker in monthly_prices.columns:
        stock_next = monthly_returns[ticker].get(next_month, np.nan)
        if hasattr(stock_next, 'item'):
            stock_next = stock_next.item()
        row[f"{ticker}_target"] = 1 if stock_next > spy_next else 0

    target_list.append(row)

target_df = pd.DataFrame(target_list).set_index("date")
print(f"Target shape: {target_df.shape}")
target_df.head()

Target shape: (77, 20)


,AAPL_target,ABT_target,AMZN_target,BAC_target,C_target,COP_target,CVX_target,EOG_target,GOOGL_target,GS_target,JNJ_target,JPM_target,META_target,MRK_target,MSFT_target,PFE_target,SLB_target,UNH_target,WFC_target,XOM_target
date,,,,,,,,,,,,,,,,,,,,
2018-07-31,1,0,1,0,0,0,0,0,0,0,0,0,0,1,1,1,0,1,0,0
2018-08-31,0,1,0,0,1,1,1,1,0,0,1,0,0,1,1,1,0,0,0,1
2018-09-30,1,1,0,1,0,0,0,0,0,1,1,1,0,1,1,1,0,1,1,1
2018-10-31,0,1,1,1,0,0,1,0,0,0,1,1,0,1,1,1,0,1,1,0
2018-11-30,0,1,0,0,0,1,1,0,1,0,0,0,1,1,1,1,0,0,0,0


## 6. Combine & Clean

In [34]:
# Merge features and targets on date
combined_df = features_df.join(target_df, how="inner")
print(f"Combined shape: {combined_df.shape}")

# Drop any remaining NaNs
combined_df = combined_df.dropna()
print(f"After dropping NaNs: {combined_df.shape}")
combined_df.head()

# Quick sanity check — target distribution (should be roughly 50/50)
target_cols = [c for c in combined_df.columns if c.endswith("_target")]
target_means = combined_df[target_cols].mean()
print(f"Average % of months each stock beat SPY: {target_means.mean():.2%}")
print(f"Min: {target_means.min():.2%}  Max: {target_means.max():.2%}")

Combined shape: (77, 140)
After dropping NaNs: (77, 140)
Average % of months each stock beat SPY: 46.95%
Min: 36.36%  Max: 61.04%


## 7. Save to Disk

In [36]:
combined_df.to_csv(f"{DATA_DIR}/features.csv")
print(f"Saved features.csv — shape: {combined_df.shape}")

Saved features.csv — shape: (77, 140)
